Script to select transcripts from the previous version of the transcriptome that don't overlap with anything from the new version and add them to the new transcriptome

In [66]:
import pandas as pd
import csv

# gtf that was created after splitting chimeras off
gtf_new = pd.read_csv('sponge_chimer_human_rb_e30_curated_250220.gtf',sep="\t",names=["scaffold", "source", "type", "start","end", " ","strand","","description"])

# gtf of the old version of the transcriptome
gtf_old = pd.read_csv("filtered_output.gtf",sep="\t",names=["scaffold", "source", "type", "start","end", " ","strand","","description"])

# path to save the final result
sorted_V3_2_df_final_csv = 'SUB3_transcriptome.chim_e30_curated.merged.renamed_250220.gtf'

def gtf_to_df(gtf_file):

    gtf_file["length"] = abs(gtf_file["end"]-gtf_file["start"] + 1)
    gtf_file[["gid","tid"]] = gtf_file["description"].str.split(";", n=1, expand=True)
    del gtf_file['source']
    del gtf_file[' ']
    del gtf_file['']
    del gtf_file['description']
    gtf_file[["","gene_id"]] = gtf_file["gid"].str.split(" ",expand=True)
    del gtf_file['gid']
    del gtf_file['']
    gtf_file[["",".","transcript_id"]] = gtf_file["tid"].str.split(" ",expand=True)
    gtf_file["transcript_id"] = gtf_file["transcript_id"].str.strip(";")
    gtf_file["transcript_id"] = gtf_file["transcript_id"].str.strip("\"")
    gtf_file["gene_id"] = gtf_file["gene_id"].str.strip("\"")
    del gtf_file['tid']
    del gtf_file['']
    del gtf_file['.']
    df = gtf_file
    return(df)

new_transcriptome = gtf_to_df(gtf_new)
old_transcriptome = gtf_to_df(gtf_old)

In [77]:
# create collapsed df where only the information on transcripts  from the old transcriptome is contained
df = old_transcriptome.drop_duplicates(subset=['transcript_id', 'start', 'end'])
df.merge(old_transcriptome[["transcript_id","scaffold","strand","type","length","gene_id"]],how="inner",on="transcript_id")

collapsed_df = df.drop_duplicates(subset='transcript_id')
collapsed_df = collapsed_df[new_transcriptome.columns.to_list()]

### Merge new transcriptome with genes from V2 that don't overlap any of the V2 transcripts

In [78]:
# Function to check if two ranges overlap
def is_overlap(row, df):
    overlaps = (
        (row["start"] <= df["end"]) & (row["end"] >= df["start"]) & (row["scaffold"] == df["scaffold"])
    )
    return overlaps.any()

# Identify genes in old transcriptome that do not overlap with any gene in the new one
non_overlapping_genes = collapsed_df[~collapsed_df.apply(lambda row: is_overlap(row, new_transcriptome), axis=1)]

# Combine new transcriptome  with non-overlapping genes from old transcriptome
result = pd.concat([new_transcriptome, non_overlapping_genes], ignore_index=True)

result["original_tr"] = result["gene_id"].str.split(".",expand=True)[0]
result["scaffold_n"] = result["scaffold"].str.replace("sca", "", regex=True).astype(int)
# result

In [79]:
# get the count of transcripts for each merged category
result.loc[result["type"] == "transcript", "original_tr"].value_counts()

original_tr
SUB3    18131
SUB2      917
SUB4      229
Name: count, dtype: int64

In [80]:
# get the  transcript IDs of the transcripts that were added from the SUB2 so that I can add their exons to the df too
selected_sub2_transcripts = result.loc[result["original_tr"] == "SUB2", "transcript_id"].tolist()

selected_sub2_tr_exons = old_transcriptome[old_transcriptome['transcript_id'].isin(selected_sub2_transcripts) &
    ~old_transcriptome['type'].str.contains('transcript')]

# add SUB2 exon information to the 'result' df:
combined_result = pd.concat([result, selected_sub2_tr_exons], ignore_index=True)
# combined_result

In [81]:
# get only the transcripts to then sort them and add column to specify order
only_transcripts = combined_result.drop_duplicates(subset="transcript_id")
only_transcripts_sorted = only_transcripts.sort_values(by=['scaffold_n', 'start'])
only_transcripts_sorted.reset_index(inplace = True)
only_transcripts_sorted["merged_order"] = only_transcripts_sorted.index + 1
only_transcripts_sorted = only_transcripts_sorted[['transcript_id', 'merged_order']]

# add 'merged order' to the dataframe and sort the dataframe
combined_result_merged = combined_result.merge(only_transcripts_sorted[['transcript_id', 'merged_order']], on='transcript_id', how='left')

sorted_result_merged = combined_result_merged.sort_values(by=['merged_order', 'start', 'type'], ascending=[True, True, True], key=lambda col: col if col.name != 'type' else col.map({'transcript': 0, 'exon': 1}))


## Rename the transcriptome

In [82]:
# set gene order and transcript numbering
sorted_V3_2_df_genes = sorted_result_merged.drop_duplicates(subset="gene_id", ignore_index=True).copy()
sorted_V3_2_df_genes['gene_order'] = sorted_V3_2_df_genes.index + 1

sorted_V3_2_df_transcripts = sorted_result_merged.drop_duplicates(subset="transcript_id", ignore_index=True).copy()
sorted_V3_2_df_transcripts['transcript_n'] = sorted_V3_2_df_transcripts.groupby('gene_id').cumcount() + 1

# merge newly calculated gene and transcript numbers
sorted_V3_2_df_order = pd.merge(sorted_result_merged, sorted_V3_2_df_genes[['gene_id', 'gene_order']], on='gene_id', how='inner')
sorted_V3_2_df_order = pd.merge(sorted_V3_2_df_order, sorted_V3_2_df_transcripts[['transcript_id', 'transcript_n']], on='transcript_id', how='inner')

In [73]:
# finalize the resulting file
raw_gtf = pd.read_csv("transcriptome.merged.filtered.renamed.gtf",sep="\t",names=["scaffold", "source", "type","start","end", " ","strand","","description"]) # example of the previously assembled gtf

sorted_V3_2_df_order['gene_id_new'] = 'SUB3.g'+ sorted_V3_2_df_order["gene_order"].astype(str)
sorted_V3_2_df_order['transcript_id_new'] = sorted_V3_2_df_order["gene_id_new"] + '.t' + sorted_V3_2_df_order["transcript_n"].astype(str)

sorted_V3_2_df_order["description"] = 'gene_id "' + sorted_V3_2_df_order["gene_id_new"] + '"; transcript_id "' +sorted_V3_2_df_order["transcript_id_new"] + '";'

sorted_V3_2_df_order[""] = '.'
sorted_V3_2_df_order[" "] = '.'
sorted_V3_2_df_order["source"] = 'StringTie'

sorted_V3_2_df_final = sorted_V3_2_df_order[raw_gtf.columns.to_list()]
sorted_V3_2_df_final

,scaffold,source,type,start,end,,strand,,description
0,sca1,StringTie,transcript,26876,27602,.,-,.,"gene_id ""SUB3.g1""; transcript_id ""SUB3.g1.t1"";"
1,sca1,StringTie,exon,26876,27602,.,-,.,"gene_id ""SUB3.g1""; transcript_id ""SUB3.g1.t1"";"
2,sca1,StringTie,transcript,27950,29777,.,+,.,"gene_id ""SUB3.g2""; transcript_id ""SUB3.g2.t1"";"
3,sca1,StringTie,exon,27950,28402,.,+,.,"gene_id ""SUB3.g2""; transcript_id ""SUB3.g2.t1"";"
4,sca1,StringTie,exon,28562,28718,.,+,.,"gene_id ""SUB3.g2""; transcript_id ""SUB3.g2.t1"";"
...,...,...,...,...,...,...,...,...,...
169857,sca784,StringTie,exon,12582,12749,.,+,.,"gene_id ""SUB3.g15524""; transcript_id ""SUB3.g15..."
169858,sca784,StringTie,exon,12898,13027,.,+,.,"gene_id ""SUB3.g15524""; transcript_id ""SUB3.g15..."
169859,sca784,StringTie,exon,13117,13314,.,+,.,"gene_id ""SUB3.g15524""; transcript_id ""SUB3.g15..."
169860,sca784,StringTie,exon,13419,14351,.,+,.,"gene_id ""SUB3.g15524""; transcript_id ""SUB3.g15..."


In [74]:
sorted_V3_2_df_final.to_csv(sorted_V3_2_df_final_csv, sep='\t', header=False, index=False, quoting=csv.QUOTE_NONE, escapechar='\\')